# Renger 2026 Gate Calibration Workflow

**Audience**
- Julia users who already know the basic `QuantumCircuit.jl` architecture and want a compact calibration runbook for MOVE and CZ primitives.

**What this notebook covers**
- Calibrate a mediated MOVE on a compact `Q-C-R` surrogate by sweeping static coupler flux and extracting projected-unitary transfer.
- Calibrate a native CZ on a compact exact-circuit `Q1-C-Q2` testbed by sweeping flux-pulse amplitude and extracting the conditional phase.
- Summarize a reduced `MOVE -> CZ -> MOVE` remote-gate recipe from those calibrated ingredients.

**Important scope**
- This notebook is a calibration runbook for compact surrogates and reduced builders.
- It is not a pulse-calibrated five-body reproduction of the paper device.
- The frozen Renger snapshot is used as a seed and sanity check, not as validated device truth.

**Runtime profile**
- Default sweeps are intentionally coarse enough to stay interactive.
- The compact MOVE surrogate is chosen so the optimum lies inside the scan window instead of on the boundary.


## Outline

1. Activate the local package environment and load notebook plotting helpers.
2. Review the current frozen Renger seed, with emphasis on the `QB1-TC1-CR` slice.
3. Calibrate a compact mediated MOVE by sweeping static coupler flux and finding the best transfer time.
4. Check how the current frozen stage-1 `QB1-TC1-CR` seed behaves under the same MOVE-style scan.
5. Calibrate a compact exact-circuit CZ by sweeping flux-pulse amplitude and extracting the conditional phase.
6. Assemble a reduced `MOVE -> CZ -> MOVE` recipe and record the figure exports.


In [ ]:
using Pkg

function find_repo_root(start::AbstractString)
    candidates = [
        normpath(start),
        normpath(joinpath(start, "..")),
        normpath(joinpath(start, "..", "..")),
    ]

    for candidate in unique(candidates)
        project_toml = joinpath(candidate, "Project.toml")
        if isfile(project_toml)
            content = read(project_toml, String)
            occursin("QuantumCircuit", content) && return candidate
        end
    end

    error("Could not find the QuantumCircuit.jl project root. Start Jupyter from the repository or open the notebook from inside it.")
end

project_root = find_repo_root(pwd())
Pkg.activate(project_root)
Pkg.instantiate()

using CairoMakie
using QuantumCircuit

include(joinpath(project_root, "output", "jupyter-notebook", "makie_helpers.jl"))
activate_notebook_theme!()

figure_exports = Dict{Symbol, NamedTuple}()


In [ ]:
function snapshot_rows(snapshot)
    rows = NamedTuple[]
    for device_name in [:QB1, :TC1, :CR, :TC2, :QB2]
        if device_name == :CR
            push!(rows, (
                name = String(device_name),
                type = "Resonator",
                fit = "paper target",
                f01_ghz = snapshot.targets["cr_f01_ghz"],
                EC = missing,
                EJ_or_EJmax = missing,
                flux = missing,
                asymmetry = missing,
            ))
        else
            data = snapshot.devices[device_name]
            push!(rows, (
                name = String(device_name),
                type = data["type"],
                fit = data["fit_classification"],
                f01_ghz = data["f01_ghz"],
                EC = data["EC"],
                EJ_or_EJmax = data["EJmax"],
                flux = data["flux"],
                asymmetry = data["asymmetry"],
            ))
        end
    end
    rows
end

round6(x) = round(x; digits = 6)

function move_summary(best)
    return (
        best_flux = round(best.flux; digits = 6),
        best_time = round(best.best_time; digits = 3),
        transfer_probability = round(best.best_transfer; digits = 6),
        leakage = round(best.best_leakage; digits = 6),
    )
end

function cz_summary(best)
    return (
        best_delta = round(best.delta; digits = 6),
        best_time = round(best.best_time; digits = 3),
        conditional_phase_over_pi = round(best.best_phase_over_pi; digits = 6),
        phase_error_over_pi = round(best.phase_error / pi; digits = 6),
        leakage = round(best.best_leakage; digits = 6),
    )
end

function build_compact_move_testbed(; flux::Real = 0.10)
    CompositeSystem(
        Transmon(:q; EJ = 0.86, EC = 0.20, ncut = 3),
        TunableCoupler(:c; EJmax = 1.10, EC = 0.15, flux = Float64(flux), asymmetry = 0.10, ncut = 3),
        Resonator(:r; ω = 0.97, dim = 2),
        CapacitiveCoupling(:q, :c; g = 0.07),
        CapacitiveCoupling(:c, :r; g = 0.07),
    )
end

function build_compact_cz_testbed()
    system = CompositeSystem(
        Transmon(:q1; EJ = 0.9, EC = 0.2, ng = 0.0, ncut = 3),
        TunableCoupler(:c; EJmax = 2.6, EC = 0.12, flux = 0.0, asymmetry = 0.1, ng = 0.0, ncut = 3),
        Transmon(:q2; EJ = 0.95, EC = 0.2, ng = 0.0, ncut = 3),
        CircuitCapacitiveCoupling(:q1, :c; G = 0.03),
        CircuitCapacitiveCoupling(:c, :q2; G = 0.03),
    )
    return (; system, hamiltonian_spec = CircuitHamiltonianSpec(charge_cutoff = 2))
end

function with_tc1_flux(snapshot, flux::Real)
    clone = deepcopy(snapshot)
    clone.devices[:TC1]["flux"] = Float64(flux)
    return clone
end

function move_transfer_curve(trace; source_index::Integer = 3, target_index::Integer = 2)
    [abs2(U[target_index, source_index]) for U in trace.unitaries]
end

function move_leakage_curve(trace; source_index::Integer = 3)
    Float64.(collect(view(trace.leakages, source_index, :)))
end

function compact_move_population(flux, tlist)
    system = build_compact_move_testbed(; flux)
    result = evolve(system, basis_state(system; q = 1, c = 0, r = 0), tlist)
    return (
        q = population_trace(result, :q, 1),
        c = population_trace(result, :c, 1),
        r = population_trace(result, :r, 1),
    )
end

function scan_compact_move(; flux_values, tlist)
    transfer_matrix = Matrix{Float64}(undef, length(flux_values), length(tlist))
    leakage_matrix = similar(transfer_matrix)
    rows = NamedTuple[]

    for (row_index, flux) in enumerate(flux_values)
        system = build_compact_move_testbed(; flux)
        spec = subspace_spec(system; subsystem_levels = (q = [0, 1], r = [0, 1]))
        trace = projected_unitary(system, spec, tlist)
        transfer_curve = move_transfer_curve(trace)
        leakage_curve = move_leakage_curve(trace)
        transfer_matrix[row_index, :] .= transfer_curve
        leakage_matrix[row_index, :] .= leakage_curve
        best = best_move(trace; source_index = 3, target_index = 2)
        push!(rows, (
            flux = Float64(flux),
            best_time = Float64(best.time),
            best_transfer = Float64(best.transfer_probability),
            best_leakage = Float64(best.leakage),
        ))
    end

    best_index = argmax(getproperty.(rows, :best_transfer))
    return (
        flux_values = Float64.(collect(flux_values)),
        times = Float64.(collect(tlist)),
        transfer_matrix = transfer_matrix,
        leakage_matrix = leakage_matrix,
        rows = rows,
        best = rows[best_index],
    )
end

function scan_snapshot_stage1_move(snapshot; flux_values, tlist)
    rows = NamedTuple[]

    for flux in flux_values
        shifted = with_tc1_flux(snapshot, flux)
        stage = renger2026_stage1_qcr_system(shifted; qubit_ncut = 5, coupler_ncut = 4, resonator_dim = 3)
        spec = subspace_spec(stage.system; subsystem_levels = (QB1 = [0, 1], CR = [0, 1]))
        trace = projected_unitary(stage.system, spec, tlist; hamiltonian_spec = stage.hamiltonian_spec)
        best = best_move(trace; source_index = 3, target_index = 2)
        push!(rows, (
            flux = Float64(flux),
            best_time = Float64(best.time),
            best_transfer = Float64(best.transfer_probability),
            best_leakage = Float64(best.leakage),
        ))
    end

    best_index = argmax(getproperty.(rows, :best_transfer))
    return (
        flux_values = Float64.(collect(flux_values)),
        times = Float64.(collect(tlist)),
        rows = rows,
        best = rows[best_index],
    )
end

function scan_compact_cz(; delta_values, tlist)
    compact_cz = build_compact_cz_testbed()
    spec = subspace_spec(
        compact_cz.system;
        hamiltonian_spec = compact_cz.hamiltonian_spec,
        subsystem_levels = (q1 = [0, 1], q2 = [0, 1]),
        basis = :dressed_static,
    )
    flux_control = FluxControl(:cz_flux, :c, (p, t) -> p.delta)

    phase_matrix = Matrix{Float64}(undef, length(delta_values), length(tlist))
    leakage_matrix = similar(phase_matrix)
    rows = NamedTuple[]

    for (row_index, delta) in enumerate(delta_values)
        trace = projected_unitary(
            compact_cz.system,
            spec,
            tlist;
            hamiltonian_spec = compact_cz.hamiltonian_spec,
            flux_controls = [flux_control],
            params = (; delta = Float64(delta)),
        )
        phase_curve = [conditional_phase(U) / pi for U in trace.unitaries]
        leakage_curve = [maximum(view(trace.leakages, :, column)) for column in eachindex(trace.times)]
        phase_matrix[row_index, :] .= phase_curve
        leakage_matrix[row_index, :] .= leakage_curve
        best = best_cz(trace)
        push!(rows, (
            delta = Float64(delta),
            best_time = Float64(best.time),
            best_phase_over_pi = Float64(best.conditional_phase / pi),
            phase_error = Float64(best.phase_error),
            best_leakage = Float64(best.leakage),
        ))
    end

    metrics = [row.phase_error + row.best_leakage for row in rows]
    best_index = argmin(metrics)
    return (
        delta_values = Float64.(collect(delta_values)),
        times = Float64.(collect(tlist)),
        phase_matrix = phase_matrix,
        leakage_matrix = leakage_matrix,
        rows = rows,
        best = rows[best_index],
    )
end


## Step 1 - Review the current frozen Renger seed

The notebook starts from the same frozen snapshot that the reduced Renger builders use elsewhere in the repository. The point here is not to declare these values "correct" gate-calibration truth, but to make the current starting point explicit before switching to compact surrogate calibration scans.

Two details matter for the later sections:
- the central resonator target is pinned at `4.2 GHz`,
- the current `TC1` seed is only a local prior, so the notebook treats the compact MOVE and CZ scans as calibration primitives rather than full-device truth.


In [ ]:
snapshot = load_renger2026_snapshot()
parameter_rows = snapshot_rows(snapshot)

frozen_seed_summary = (
    devices = parameter_rows,
    targets = (
        cr_f01_ghz = snapshot.targets["cr_f01_ghz"],
        beta_qc_qb1 = snapshot.targets["beta_qc_qb1"],
        beta_cr = snapshot.targets["beta_cr"],
        coupler_alpha_prior_ghz = snapshot.targets["coupler_alpha_prior_ghz"],
    ),
)

display(parameter_rows)
frozen_seed_summary


## Step 2 - Calibrate a compact mediated MOVE

This section uses a deliberately small `Q-C-R` surrogate with one interior optimum in the static coupler-flux sweep. The goal is to show the calibration loop clearly:
- sweep a control parameter,
- compute a projected unitary on the logical `|10⟩, |01⟩` subspace,
- pick the best transfer time,
- inspect the population trace at the chosen operating point.

The surrogate is compact on purpose. It is meant to teach and validate the calibration machinery before applying the same logic to the heavier paper-aligned builders.


In [ ]:
move_flux_values = collect(range(0.04, 0.16; length = 13))
move_tlist = collect(range(0.0, 80.0; length = 401))

move_scan = scan_compact_move(; flux_values = move_flux_values, tlist = move_tlist)
best_compact_move = move_scan.best
move_populations = compact_move_population(best_compact_move.flux, move_tlist)

move_heatmap = heatmap_figure(
    move_scan.times,
    move_scan.flux_values,
    move_scan.transfer_matrix;
    title = "Compact MOVE calibration: transfer probability",
    xlabel = "time (ns)",
    ylabel = "coupler flux",
    colorlabel = "|U[target, source]|^2",
    colorrange = (0.0, 1.0),
)
move_heatmap_saved = save_figure(move_heatmap, project_root, "gate_calibration_move_heatmap")
figure_exports[:gate_calibration_move_heatmap] = move_heatmap_saved

move_tradeoff_fig = Figure(size = NOTEBOOK_TALL)
ax_move_transfer = Axis(
    move_tradeoff_fig[1, 1];
    title = "Best MOVE transfer by static flux",
    xlabel = "coupler flux",
    ylabel = "best transfer",
)
move_best_transfer = getproperty.(move_scan.rows, :best_transfer)
move_best_times = getproperty.(move_scan.rows, :best_time)
lines!(ax_move_transfer, move_scan.flux_values, move_best_transfer; color = :royalblue3)
scatter!(ax_move_transfer, move_scan.flux_values, move_best_transfer; color = :royalblue3, markersize = 10)
scatter!(ax_move_transfer, [best_compact_move.flux], [best_compact_move.best_transfer]; color = :black, markersize = 16)

ax_move_time = Axis(
    move_tradeoff_fig[2, 1];
    xlabel = "coupler flux",
    ylabel = "best time (ns)",
)
lines!(ax_move_time, move_scan.flux_values, move_best_times; color = :firebrick3)
scatter!(ax_move_time, move_scan.flux_values, move_best_times; color = :firebrick3, markersize = 10)
scatter!(ax_move_time, [best_compact_move.flux], [best_compact_move.best_time]; color = :black, markersize = 16)
move_tradeoff_saved = save_figure(move_tradeoff_fig, project_root, "gate_calibration_move_tradeoffs")
figure_exports[:gate_calibration_move_tradeoffs] = move_tradeoff_saved

move_population_fig = line_figure(
    [
        (x = move_populations.q.times, y = move_populations.q.values, label = "q excited", color = :royalblue3),
        (x = move_populations.c.times, y = move_populations.c.values, label = "c excited", color = :goldenrod3),
        (x = move_populations.r.times, y = move_populations.r.values, label = "r excited", color = :firebrick3),
    ];
    title = "Compact MOVE population trace at calibrated flux",
    xlabel = "time (ns)",
    ylabel = "population",
)
move_population_saved = save_figure(move_population_fig, project_root, "gate_calibration_move_population_trace")
figure_exports[:gate_calibration_move_population_trace] = move_population_saved

display(move_heatmap)
display(move_tradeoff_fig)
display(move_population_fig)
(
    move = move_summary(best_compact_move),
    max_coupler_population = round(maximum(move_populations.c.values); digits = 6),
    saved = (
        heatmap = move_heatmap_saved,
        tradeoffs = move_tradeoff_saved,
        population_trace = move_population_saved,
    ),
)


## Step 3 - Check the current frozen `QB1-TC1-CR` seed

The compact surrogate above is deliberately well-behaved. The frozen stage-1 `QB1-TC1-CR` builder is not a calibrated gate, so it is useful to look at it separately.

This section sweeps the static `TC1` flux in the reduced stage-1 effective builder and records the best transfer it can reach. If the transfer stays weak or the best time drifts far out, that is a sign that the frozen seed should be treated as a starting prior, not a ready-to-run MOVE calibration.


In [ ]:
snapshot_move_flux_values = collect(range(0.00, 0.20; length = 11))
snapshot_move_tlist = collect(range(0.0, 400.0; length = 401))

snapshot_move_scan = scan_snapshot_stage1_move(snapshot; flux_values = snapshot_move_flux_values, tlist = snapshot_move_tlist)
best_snapshot_move = snapshot_move_scan.best

snapshot_move_fig = Figure(size = NOTEBOOK_TALL)
ax_snapshot_transfer = Axis(
    snapshot_move_fig[1, 1];
    title = "Frozen stage-1 QB1-TC1-CR seed: best MOVE transfer",
    xlabel = "TC1 flux",
    ylabel = "best transfer",
)
snapshot_best_transfer = getproperty.(snapshot_move_scan.rows, :best_transfer)
snapshot_best_times = getproperty.(snapshot_move_scan.rows, :best_time)
lines!(ax_snapshot_transfer, snapshot_move_scan.flux_values, snapshot_best_transfer; color = :royalblue3)
scatter!(ax_snapshot_transfer, snapshot_move_scan.flux_values, snapshot_best_transfer; color = :royalblue3, markersize = 10)
scatter!(ax_snapshot_transfer, [best_snapshot_move.flux], [best_snapshot_move.best_transfer]; color = :black, markersize = 16)

ax_snapshot_time = Axis(
    snapshot_move_fig[2, 1];
    xlabel = "TC1 flux",
    ylabel = "best time (ns)",
)
lines!(ax_snapshot_time, snapshot_move_scan.flux_values, snapshot_best_times; color = :firebrick3)
scatter!(ax_snapshot_time, snapshot_move_scan.flux_values, snapshot_best_times; color = :firebrick3, markersize = 10)
scatter!(ax_snapshot_time, [best_snapshot_move.flux], [best_snapshot_move.best_time]; color = :black, markersize = 16)

snapshot_move_saved = save_figure(snapshot_move_fig, project_root, "gate_calibration_snapshot_stage1_move_scan")
figure_exports[:gate_calibration_snapshot_stage1_move_scan] = snapshot_move_saved

display(snapshot_move_fig)
(
    frozen_stage1_move = move_summary(best_snapshot_move),
    note = "Low transfer or very long best times mean the frozen seed is not yet a calibrated MOVE operating point.",
    saved = snapshot_move_saved,
)


## Step 4 - Calibrate a compact exact-circuit CZ

This section reuses the repository's exact-circuit two-qubit-plus-coupler teaching model, but turns the single-point example into a scan over flux-pulse amplitude `delta`.

The workflow is the same one used in the codebase:
- define a dressed logical subspace,
- apply a flux control to the coupler,
- project the resulting unitary into the logical basis,
- evaluate the conditional phase and leakage,
- pick the best operating point.


In [ ]:
cz_delta_values = collect(range(0.02, 0.08; length = 13))
cz_tlist = collect(range(0.0, 120.0; length = 121))

cz_scan = scan_compact_cz(; delta_values = cz_delta_values, tlist = cz_tlist)
best_compact_cz = cz_scan.best

cz_heatmap_fig = Figure(size = (1080, 920))
ax_phase = Axis(
    cz_heatmap_fig[1, 1];
    title = "Compact CZ calibration: conditional phase / pi",
    xlabel = "time (ns)",
    ylabel = "flux-pulse delta",
)
phase_hm = image!(
    ax_phase,
    (minimum(cz_scan.times), maximum(cz_scan.times)),
    (minimum(cz_scan.delta_values), maximum(cz_scan.delta_values)),
    cz_scan.phase_matrix;
    colormap = :balance,
    colorrange = (-1.0, 1.0),
    interpolate = false,
)
Colorbar(cz_heatmap_fig[1, 2], phase_hm; label = "conditional phase / pi")
scatter!(ax_phase, [best_compact_cz.best_time], [best_compact_cz.delta]; color = :black, markersize = 16)

ax_leakage = Axis(
    cz_heatmap_fig[2, 1];
    title = "Compact CZ calibration: max leakage",
    xlabel = "time (ns)",
    ylabel = "flux-pulse delta",
)
leakage_hm = image!(
    ax_leakage,
    (minimum(cz_scan.times), maximum(cz_scan.times)),
    (minimum(cz_scan.delta_values), maximum(cz_scan.delta_values)),
    cz_scan.leakage_matrix;
    colormap = :viridis,
    interpolate = false,
)
Colorbar(cz_heatmap_fig[2, 2], leakage_hm; label = "max leakage")
scatter!(ax_leakage, [best_compact_cz.best_time], [best_compact_cz.delta]; color = :white, markersize = 16)

cz_heatmap_saved = save_figure(cz_heatmap_fig, project_root, "gate_calibration_cz_heatmaps")
figure_exports[:gate_calibration_cz_heatmaps] = cz_heatmap_saved

cz_tradeoff_fig = Figure(size = NOTEBOOK_TALL)
ax_cz_phase = Axis(
    cz_tradeoff_fig[1, 1];
    title = "Best CZ operating point by pulse delta",
    xlabel = "flux-pulse delta",
    ylabel = "best conditional phase / pi",
)
cz_best_phase = getproperty.(cz_scan.rows, :best_phase_over_pi)
lines!(ax_cz_phase, cz_scan.delta_values, cz_best_phase; color = :royalblue3)
scatter!(ax_cz_phase, cz_scan.delta_values, cz_best_phase; color = :royalblue3, markersize = 10)
scatter!(ax_cz_phase, [best_compact_cz.delta], [best_compact_cz.best_phase_over_pi]; color = :black, markersize = 16)

ax_cz_error = Axis(
    cz_tradeoff_fig[2, 1];
    xlabel = "flux-pulse delta",
    ylabel = "phase error / pi or leakage",
)
cz_phase_error = getproperty.(cz_scan.rows, :phase_error) ./ pi
cz_best_leakage = getproperty.(cz_scan.rows, :best_leakage)
lines!(ax_cz_error, cz_scan.delta_values, cz_phase_error; color = :firebrick3, label = "phase error / pi")
scatter!(ax_cz_error, cz_scan.delta_values, cz_phase_error; color = :firebrick3, markersize = 10)
lines!(ax_cz_error, cz_scan.delta_values, cz_best_leakage; color = :goldenrod3, label = "leakage")
scatter!(ax_cz_error, cz_scan.delta_values, cz_best_leakage; color = :goldenrod3, markersize = 10)
axislegend(ax_cz_error; position = :rt)
scatter!(ax_cz_error, [best_compact_cz.delta], [best_compact_cz.phase_error / pi]; color = :black, markersize = 16)

cz_tradeoff_saved = save_figure(cz_tradeoff_fig, project_root, "gate_calibration_cz_tradeoffs")
figure_exports[:gate_calibration_cz_tradeoffs] = cz_tradeoff_saved

display(cz_heatmap_fig)
display(cz_tradeoff_fig)
(
    cz = cz_summary(best_compact_cz),
    saved = (
        heatmaps = cz_heatmap_saved,
        tradeoffs = cz_tradeoff_saved,
    ),
)


## Step 5 - Assemble a reduced remote-gate recipe

With one calibrated MOVE surrogate and one calibrated CZ surrogate in hand, we can make the remote-gate structure explicit without pretending the result is already a full five-body hardware calibration.

The main use of this section is bookkeeping: it shows how the calibrated primitive times and dominant quality metrics stack into the logical `MOVE -> CZ -> MOVE` workflow.


In [ ]:
remote_gate_recipe = [
    (
        step = 1,
        operation = "MOVE(QB1, CR)",
        model = "compact Q-C-R surrogate",
        time_ns = round(best_compact_move.best_time; digits = 3),
        primary_metric = round(best_compact_move.best_transfer; digits = 6),
    ),
    (
        step = 2,
        operation = "CZ(QB2, CR)",
        model = "compact exact-circuit surrogate",
        time_ns = round(best_compact_cz.best_time; digits = 3),
        primary_metric = round(best_compact_cz.best_phase_over_pi; digits = 6),
    ),
    (
        step = 3,
        operation = "MOVE(QB1, CR)",
        model = "compact Q-C-R surrogate",
        time_ns = round(best_compact_move.best_time; digits = 3),
        primary_metric = round(best_compact_move.best_transfer; digits = 6),
    ),
]

remote_gate_summary = (
    estimated_total_sequence_time_ns = round(2 * best_compact_move.best_time + best_compact_cz.best_time; digits = 3),
    move_transfer = round(best_compact_move.best_transfer; digits = 6),
    cz_conditional_phase_over_pi = round(best_compact_cz.best_phase_over_pi; digits = 6),
    loose_leakage_budget = round(min(1.0, 2 * best_compact_move.best_leakage + best_compact_cz.best_leakage); digits = 6),
    figure_exports = figure_exports,
    note = "This is a reduced calibration recipe, not a five-body pulse-optimized remote gate.",
)

display(remote_gate_recipe)
remote_gate_summary


## Next steps

- Replace the compact MOVE surrogate with a better-behaved exact-circuit `QB1-TC1-CR` slice once the frozen `TC1` seed and couplings are re-fit.
- Add denser convergence checks for `ncut`, resonator dimension, and `charge_cutoff` before trusting any absolute timing claim.
- If you want paper-faithful gate numbers, move from these static/coarse sweeps to shaped flux pulses and the full reduced five-body circuit.
